# 04 - Proposal-Style Lab Report Demo

**ChatGPT Track**  
**allen-lab-report-tool**

This notebook turns earlier artifacts into a readable proposal-style lab report demo.

```text
context
+ source metadata
+ report sections
→ proposal-style continuation report
```

This notebook writes local artifacts only. It does not push to GitHub.


In [ ]:
# ================================================
# SETUP: Colab + local
# ================================================
from pathlib import Path
import json
import sys
import subprocess
import os
from datetime import datetime, timezone

REPO_NAME = "allen-lab-report-tool"
REPO_URL = "https://github.com/thinkthoughts/allen-lab-report-tool.git"

cwd = Path.cwd()

if (cwd / "src" / "chatgpt" / "lab_context.py").exists():
    repo_root = cwd
elif cwd.name == "chatgpt" and cwd.parent.name == "notebooks":
    repo_root = cwd.parents[1]
elif (cwd / REPO_NAME / "src" / "chatgpt" / "lab_context.py").exists():
    repo_root = cwd / REPO_NAME
else:
    print("Repo not found in current runtime. Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
    repo_root = cwd / REPO_NAME

src_path = repo_root / "src"
results_dir = repo_root / "results" / "chatgpt"
reports_dir = repo_root / "reports" / "chatgpt"
docs_dir = repo_root / "docs" / "chatgpt"

results_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)
docs_dir.mkdir(parents=True, exist_ok=True)

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("cwd:", cwd)
print("repo_root:", repo_root)
print("results_dir:", results_dir)
print("reports_dir:", reports_dir)
print("docs_dir:", docs_dir)


## 1. Ensure Required Artifacts Exist

Notebook 04 expects artifacts from Notebooks 01-03 or from `src/chatgpt/build_artifacts.py`.


In [ ]:
required_paths = [
    results_dir / "allen_lab_context.json",
    results_dir / "source_metadata.json",
    results_dir / "report_sections.json",
]

missing = [p for p in required_paths if not p.exists()]

if missing:
    print("Missing artifacts:")
    for p in missing:
        print(" -", p)

    builder_path = repo_root / "src" / "chatgpt" / "build_artifacts.py"
    if builder_path.exists():
        print("Running ChatGPT artifact builder...")
        env = os.environ.copy()
        env["PYTHONPATH"] = str(src_path)
        subprocess.run(
            ["python3", str(builder_path)],
            cwd=repo_root,
            env=env,
            check=True,
        )
    else:
        raise FileNotFoundError(
            "Missing artifacts and no build_artifacts.py found. "
            "Run Notebooks 01-03 or add src/chatgpt/build_artifacts.py."
        )

for p in required_paths:
    print("✓", p, p.exists())


## 2. Load Artifacts

In [ ]:
context_path = results_dir / "allen_lab_context.json"
metadata_path = results_dir / "source_metadata.json"
sections_path = results_dir / "report_sections.json"

allen_context = json.loads(context_path.read_text(encoding="utf-8"))
source_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
report_sections = json.loads(sections_path.read_text(encoding="utf-8"))

source_record = source_metadata["source_record"]
context_matches = source_metadata["context_matches"]
sections = report_sections["sections"]

print("Institution:", allen_context["institution"])
print("Source title:", source_record["title"])
print("Sections:", len(sections))


## 3. Demo Framing

This notebook presents the ChatGPT track as a readable demo rather than only an artifact builder.


In [ ]:
demo_frame = {
    "demo_title": "Allen Lab Report-Tool Continuation Demo",
    "core_idea": (
        "Use a context-aware report pipeline to translate Allen Lab-style source materials "
        "into structured, reproducible, reviewable lab reports."
    ),
    "delivery_mode": "proposal-style white paper / lab handoff artifact",
    "notebook_role": "assemble readable report from generated artifacts",
}

demo_frame


## 4. Assemble Proposal-Style Sections

In [ ]:
def bullets(items):
    if not items:
        return "- (none listed)"
    return "\n".join(f"- {item}" for item in items)

focus = context_matches.get("matched_focus_areas", [])
platforms = context_matches.get("matched_equipment_or_platforms", [])
priorities = context_matches.get("matched_report_priorities", [])

proposal_sections = [
    {
        "title": "Executive Summary",
        "content": (
            "This demo presents a context-aware lab report pipeline for Allen Lab-style work. "
            "The pipeline starts with an institutional context profile, attaches source metadata, "
            "generates structured report sections, and exports reviewable Markdown and JSON artifacts."
        ),
    },
    {
        "title": "Research Proposal Frame",
        "content": (
            "The working proposal is to use reproducible report tooling to continue scientific outputs "
            "as structured, inspectable lab reports. The report tool organizes source context, methods "
            "vocabulary, matched priorities, and follow-up questions so a reviewer can quickly decide "
            "what to keep, revise, or extend."
        ),
    },
    {
        "title": "Source Under Review",
        "content": f"""**Title:** {source_record['title']}

**Source type:** {source_record['source_type']}

**Institution:** {source_record['institution']}

**Summary:** {source_record['abstract_or_summary']}
""",
    },
    {
        "title": "Allen Lab Alignment",
        "content": f"""Matched focus areas:

{bullets(focus)}

Matched equipment or platforms:

{bullets(platforms)}

Matched report priorities:

{bullets(priorities)}
""",
    },
    {
        "title": "Methods Direction",
        "content": (
            "The next technical step is to replace placeholder metadata with a specific Allen Lab paper, "
            "dataset page, methods page, or source text. Once a concrete source is selected, the pipeline "
            "can extract methods terms, figure/table references, dataset provenance, and reproducibility notes."
        ),
    },
    {
        "title": "Handoff Value",
        "content": (
            "The deliverable is not a repo tour. The deliverable is a reviewable report artifact: "
            "a lab-context profile, a source metadata record, context-aware report sections, and a "
            "proposal-style continuation report that can be corrected or extended by a lab member."
        ),
    },
    {
        "title": "Next Steps",
        "content": """- Replace the placeholder source with one specific Allen Lab source.
- Add source URL, title, authors, methods summary, and dataset links.
- Generate a first complete proposal/report artifact.
- Compare ChatGPT, Grok, and human-edited versions.
- Prepare a one-page white paper and optional demo notebook for in-person delivery.
""",
    },
]

proposal_sections


## 5. Display Proposal Section Index

In [ ]:
import pandas as pd

proposal_df = pd.DataFrame([
    {"section": i + 1, "title": section["title"]}
    for i, section in enumerate(proposal_sections)
])

display(proposal_df)


## 6. Build Full Demo Artifact

In [ ]:
full_report_artifact = {
    "generator_track": "chatgpt",
    "source_file": "notebooks/chatgpt/04_proposal_style_lab_report_demo.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "demo_frame": demo_frame,
    "inputs": {
        "allen_context": str(context_path.relative_to(repo_root)),
        "source_metadata": str(metadata_path.relative_to(repo_root)),
        "report_sections": str(sections_path.relative_to(repo_root)),
    },
    "source_title": source_record["title"],
    "proposal_sections": proposal_sections,
}

full_report_artifact


## 7. Export JSON

In [ ]:
full_json_path = results_dir / "full_lab_report_demo.json"

with full_json_path.open("w", encoding="utf-8") as f:
    json.dump(full_report_artifact, f, indent=2, ensure_ascii=False)

print("Wrote:", full_json_path)


## 8. Export Markdown

In [ ]:
section_blocks = []
for section in proposal_sections:
    section_blocks.append(f"## {section['title']}\n\n{section['content'].strip()}\n")

full_md = f"""# Allen Lab Report-Tool Continuation Demo

**Generator track:** ChatGPT  
**Notebook:** `notebooks/chatgpt/04_proposal_style_lab_report_demo.ipynb`  
**Source title:** {source_record['title']}

## Demo Purpose

{demo_frame['core_idea']}

## Inputs

- `{str(context_path.relative_to(repo_root))}`
- `{str(metadata_path.relative_to(repo_root))}`
- `{str(sections_path.relative_to(repo_root))}`

---

{chr(10).join(section_blocks)}
"""

full_md_path = reports_dir / "full_lab_report_demo.md"
full_md_path.write_text(full_md, encoding="utf-8")

docs_md_path = docs_dir / "full_lab_report_demo.md"
docs_md_path.write_text(full_md, encoding="utf-8")

print("Wrote:", full_md_path)
print("Wrote:", docs_md_path)


## 9. Confirm Export

In [ ]:
print(full_md_path.read_text(encoding="utf-8"))


## 10. Summary

Notebook 04 creates the first demo-readable ChatGPT report artifact.

Outputs:

```text
results/chatgpt/full_lab_report_demo.json
reports/chatgpt/full_lab_report_demo.md
docs/chatgpt/full_lab_report_demo.md
```

The next notebook can compare versions or prepare a white-paper/proposal bundle.
